# P86 — La competición M4: resultados, hallazgos, conclusiones y camino a seguir

## 1. Título y paper

**Paper:** *The M4 Competition: Results, findings, conclusion and way forward*  
**Autoría:** Spyros Makridakis, Evangelos Spiliotis, Vassilios Assimakopoulos  
**Año y venue:** 2018 · International Journal of Forecasting, 34(4), 802–808  
**Nivel:** L3 · **Motor:** `m4`  
**Ficha completa:** [`P86_m4`](../../papers/foundational/P86_m4/README.md)

**Hito:** Cien mil series y sesenta y un métodos para responder empíricamente qué funciona al predecir series temporales — y la respuesta incomoda a todo el mundo.

- [doi:10.1016/j.ijforecast.2018.06.001](https://doi.org/10.1016/j.ijforecast.2018.06.001)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Cada artículo de predicción reportaba mejoras sobre sus propias series y sus propias líneas base. Sin una evaluación común y a ciegas, el campo no podía saber qué funcionaba de verdad.
2. Ejecutar una implementación mínima de la propuesta: Una competición abierta con 100 000 series reales de dominios distintos, horizontes fijos, métricas declaradas de antemano y evaluación fuera de muestra sobre datos que los participantes no ven.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Makridakis y Hibon (2000), la competición M3
- P76


## 4. Intuición

Cien mil series reales, sesenta y un métodos, evaluación a ciegas y métricas declaradas de antemano. Es lo más parecido a un experimento controlado que ha tenido la predicción de series temporales, y su resultado incomoda a todo el mundo.


## 5. Concepto mínimo

```text
Dentro de muestra : ajustar el pasado. Un polinomio de grado alto siempre gana.
Fuera de muestra  : predecir lo que no se ha visto. Ahí gana otra cosa.

El error de todo backtesting mal hecho es evaluar con datos que el modelo
ya vio, aunque sea indirectamente al elegir hiperparámetros.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('m4', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué método ajusta mejor dentro de muestra?
2. ¿Y fuera de muestra?
3. ¿Dónde queda la combinación de tres métodos?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('m4', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('m4', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El polinomio de grado 11 ajusta mejor dentro de muestra (MAE 6,31 frente a 7,71 del lineal) y fuera de muestra se dispara a **6 983**. El mejor fuera de muestra es el polinomio de grado 1, con 7,48. Y la combinación de tres métodos queda **3ª de 7**: no gana, y tampoco se hunde.


## 10. Comentario pedagógico

El hallazgo de la M4 sobre 100 000 series es exactamente ese perfil: la combinación rara vez es la mejor y casi nunca es la peor, así que en ausencia de información sobre la serie concreta es la apuesta razonable. Y las líneas base ingenuas no son un trámite: son el listón que un método sofisticado tiene que superar **fuera de muestra** antes de haber demostrado nada.


## 11. Error o anti-patrón deliberado

Anti-patrón: elegir el modelo por su ajuste al histórico.


In [ ]:
print('Un polinomio de grado alto reproduce el pasado casi exactamente.')
print('Y extrapola a cualquier cosa: en esta serie, un MAE de 6983 a doce pasos.')
print('Ajustar el pasado y predecir el futuro son objetivos distintos y a menudo opuestos.')

## 12. Corrección

El protocolo que separa una cosa de la otra:


In [ ]:
r = run_paper_lab('m4', seed=7)['result']
print('ajuste DENTRO de muestra:', r['ajuste_dentro_de_muestra'])
print()
for fila in r['resultados_fuera_de_muestra']:
    print(f"  {fila['metodo']:<22} MAE fuera = {fila['mae_fuera_de_muestra']}")

## 13. Desafío guiado

Compara el orden de los polinomios dentro y fuera de muestra, y explica por qué el ranking se invierte.


In [ ]:
r = run_paper_lab('m4', seed=3)['result']
show(r)

## 14. Desafío autónomo

Coge una serie de tu trabajo y evalúala con validación en ventanas deslizantes: varios cortes, mismo horizonte. Compara tu método favorito contra el ingenuo estacional y contra la combinación de tres métodos simples.


## 15. Evidencia de aprendizaje

Guarda las dos tablas —dentro y fuera de muestra— y tu protocolo de backtesting con el número de ventanas y el horizonte.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P86_m4/README.md) · evaluación formal: [`assessments/papers/P86_m4.md`](../../assessments/papers/P86_m4.md)


## 16. Cierre

Aquí se cierra la ruta clásica: agrupar, dividir, separar, regularizar, combinar y, sobre todo, medir fuera de muestra. Lo que sigue en el programa es lo que ocurre cuando el modelo aprende también la representación.


## 17. Conexión con el siguiente hito

- P62

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
